<a href="https://colab.research.google.com/github/pris25123/BoundedByLLMs-FineTuningSLM/blob/main/NLP_orangeproblem_BoundedByLLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Multimodal Fine-Tuning with Small Language Models
Bunded By LLMs
PRIYANKA S - PES1UG23AM219

PADARTHI NEHA SAI - PES1UG23AM200

Dataset:
RICO-Screen2Words
https://huggingface.co/datasets/rootsautomation/RICO-Screen2Words


Model:

Training Strategy:
To ensure the training runs smoothly on T4 GPU, we:

Pipeline:


## Cell 1: Install Dependencies

We install the core libraries needed for this project:
- `transformers`: Provides the pretrained ViT-GPT2 model, tokenizer, and Trainer API
- `datasets`: Used to load the RICO Screen2Words dataset directly from HuggingFace Hub
- `pillow`: Image processing — converts screenshots to RGB format for the feature extractor
- `accelerate`: Required backend for HuggingFace Trainer to handle device placement and fp16 training
- `huggingface_hub`: Enables `notebook_login()` and `push_to_hub()` for uploading the fine-tuned model
- `scikit-learn`: Used only for `train_test_split` to create reproducible train/val splits

In [ ]:
# Cell-1
!pip install -q transformers datasets pillow accelerate huggingface_hub scikit-learn

In [ ]:
# Cell-2
from huggingface_hub import notebook_login

notebook_login()

## Cell 3: Load the RICO Screen2Words Dataset

We use the `rootsautomation/RICO-Screen2Words` dataset from HuggingFace Hub.
- **RICO** (Rico Is a Collection of UIs) is a large-scale dataset of Android UI screenshots
- **Screen2Words** extends RICO with 5 human-written captions per screenshot, making it
  well-suited for image captioning tasks on mobile app interfaces
- The dataset is loaded in its default splits (train/validation/test)
- We will only use the `train` split and manually create our own val split due to the
  small subset size we are working with

In [ ]:
# Cell-3
from datasets import load_dataset

dataset = load_dataset("rootsautomation/RICO-Screen2Words")

print(dataset)

## Cell 4: Visualize a Sample

We display one image from the training set alongside its captions to verify:
1. The dataset loaded correctly
2. The image format is as expected (mobile UI screenshot)
3. The captions are human-readable natural language descriptions

Each sample contains a PIL Image and a list of 5 reference captions.
This step is purely exploratory and does not affect training.

In [ ]:
# Cell-4
import matplotlib.pyplot as plt

sample = dataset["train"][0]

plt.imshow(sample["image"])
plt.axis("off")
plt.title("Sample Image")
print("Caption:", sample["captions"])
plt.show()


## Cell 5: Load Pretrained ViT-GPT2 Model

We use `nlpconnect/vit-gpt2-image-captioning` as our base model. This is a
VisionEncoderDecoder architecture combining:
- **Encoder**: Vision Transformer (ViT) — extracts visual features from the screenshot
- **Decoder**: GPT-2 — generates natural language captions from those features

**Why this model?**
- It is a Small Language Model (SLM) that fits comfortably within T4 GPU memory (~3.5GB)
- It is pretrained on general image captioning (COCO), giving it a strong visual-language
  foundation before we fine-tune it on mobile UI images
- The VisionEncoderDecoder architecture is well-supported by HuggingFace Trainer,
  simplifying the training loop

**`tokenizer.pad_token = tokenizer.eos_token`** — GPT-2 has no dedicated padding token by
default. We set it to the EOS token so the tokenizer can pad sequences to a fixed length
during batch processing. This is the standard workaround for GPT-2-based decoders.
This is also why the attention mask warning appears during inference — the model cannot
distinguish pad from EOS by token ID alone, so passing `attention_mask` explicitly is
recommended when this matters.

The model is moved to CUDA (`model.to("cuda")`) to use the T4 GPU for both training
and inference.

In [ ]:
# Cell-5
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
import torch

model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
feature_extractor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model = model.to("cuda")
print("Model loaded ✓")

## Cell 6: Split Dataset and Preprocess

**Why 500 samples?**

We select the first 500 samples from the training set rather than the full dataset (~22,000
images). This keeps training time within ~10–15 minutes on a T4 GPU, making the notebook
fully reproducible on free-tier Colab or Kaggle without timeouts. It also reduces the risk
of out-of-memory errors during preprocessing.

**80/20 train/val split**
We use an 80/20 split (400 train, 100 val) with `random_state=42` for reproducibility.
A fixed random seed ensures the same split is produced on every run, making results
comparable across experiments.

**Preprocessing decisions:**
- `feature_extractor`: Converts the PIL image to a normalized pixel tensor expected by ViT
  (resized to 224×224, normalized with ImageNet mean/std). This is handled automatically
  by `ViTImageProcessor`.
- `caption = example["captions"][0]`: We use only the first of the 5 available captions
  per image during training. Using all 5 would require a multi-reference training setup;
  a single caption keeps the pipeline simple without significantly hurting performance.
- `max_length=32`: Mobile UI captions in Screen2Words are short, typically under 15 words.
  A max length of 32 tokens gives enough headroom for any caption in the dataset while
  keeping memory usage low during batch tokenization.
- `padding="max_length"`: All sequences are padded to exactly 32 tokens so they can be
  stacked into batches of uniform shape.
- `labels = [-100 if t == pad_token_id else t]`: Padding positions in the labels are
  masked with -100. PyTorch's CrossEntropyLoss ignores index -100 by default, so the
  model is not penalized for predicting padding tokens — only real caption tokens
  contribute to the loss.

In [ ]:
# Cell-6: Split and Preprocess
from sklearn.model_selection import train_test_split
import numpy as np

# Using only 200 samples to keep it light on T4
small = dataset["train"].select(range(500))
train_idx, val_idx = train_test_split(range(len(small)), test_size=0.2, random_state=42)
train_dataset = small.select(train_idx)
val_dataset   = small.select(val_idx)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

def preprocess(example):
    img = example["image"].convert("RGB")

    pixel_values = feature_extractor(
        images=img,
        return_tensors="pt"
    ).pixel_values.squeeze(0).numpy()

    caption = example["captions"][0] if isinstance(example["captions"], list) else example["captions"]

    tokenized = tokenizer(caption, padding="max_length", truncation=True, max_length=32)

    # Mask padding tokens
    labels = [-100 if t == tokenizer.pad_token_id else t for t in tokenized["input_ids"]]

    return {"pixel_values": pixel_values, "labels": labels}

train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(preprocess,   remove_columns=val_dataset.column_names)
print("Columns:", train_dataset.column_names)  # ['pixel_values', 'labels']

## Cell 7: Collate Function and Sanity Check

**`collate_fn`** assembles individual preprocessed samples into a batch:
- `pixel_values`: Stacked into a float32 tensor of shape `(batch_size, 3, 224, 224)`
- `labels`: Stacked into a long int tensor of shape `(batch_size, 32)`

We use `np.stack` explicitly because HuggingFace datasets return pixel values as numpy
arrays after the `.map()` preprocessing step. The `float()` cast ensures compatibility
with fp16 training.

The sanity check prints the shape of a 2-sample batch to confirm that preprocessing
and collation are working correctly before committing to a full training run. This
catches shape mismatches early and avoids wasted GPU time.

In [ ]:
# Cell-7: Collate and Sanity Check
import torch
import numpy as np

def collate_fn(batch):
    pixel_values = torch.from_numpy(np.stack([np.array(x["pixel_values"]) for x in batch])).float()
    labels       = torch.tensor([x["labels"] for x in batch], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

# Sanity check
out = collate_fn([train_dataset[i] for i in range(2)])
print({k: v.shape for k, v in out.items()})

## Cell 8: Fine-Tuning with HuggingFace Trainer

We use the HuggingFace `Trainer` API to fine-tune the ViT-GPT2 model on our
preprocessed RICO subset.

**Hyperparameter decisions:**

- **`per_device_train_batch_size=4`**: Batch size of 4 keeps GPU memory usage under
  ~8GB on a T4 (16GB VRAM), leaving headroom for the optimizer states and gradients.
  Larger batches would risk OOM errors on free-tier hardware.

- **`num_train_epochs=5`**: Five epochs over 400 samples = 2,000 gradient steps total.
  This is enough for the model to adapt its decoder to the mobile UI captioning domain
  without overfitting on such a small dataset.

- **`learning_rate=3e-5`**: Slightly lower than the Trainer default (5e-5) to avoid
  catastrophic forgetting of the pretrained COCO captioning knowledge. A lower lr
  fine-tunes the model gently toward the UI domain rather than overwriting its
  general visual-language representations.

- **`warmup_steps=20`**: A short linear warmup over the first 20 steps prevents a
  large initial gradient update from destabilizing the pretrained weights at the
  start of training.

- **`fp16=True`**: Mixed-precision (16-bit float) training halves memory usage and
  speeds up training by ~1.5–2× on T4 GPUs, which have dedicated Tensor Cores for fp16.

- **`save_strategy="no"`**: We skip checkpoint saving during training to avoid writing
  large files to disk on Colab's limited storage. The final model is saved explicitly
  after training completes (Cell 9).

- **`report_to="none"`**: Disables Weights & Biases and other experiment trackers
  to keep the notebook self-contained without requiring external API keys.

- **`logging_steps=10`**: Logs training loss every 10 steps, providing enough
  granularity to detect divergence without flooding the output.

In [ ]:
# Cell-8: Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./rico-model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,          # more epochs
    learning_rate=3e-5,          # slightly lower lr
    warmup_steps=20,             # smooth startup
    fp16=True,
    report_to="none",
    save_strategy="no",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

trainer.train()
print("Training done ✓")

##Saving the model locally

In [ ]:
# Cell-9: Save Model Locally
model.save_pretrained("rico-vit-gpt2-finetuned")
feature_extractor.save_pretrained("rico-vit-gpt2-finetuned")
tokenizer.save_pretrained("rico-vit-gpt2-finetuned")
print("Saved locally ✓")

In [ ]:
import shutil
from google.colab import files

# Zip the folder
shutil.make_archive("rico-vit-gpt2-finetuned", "zip", ".", "rico-vit-gpt2-finetuned")
print("Zipped ✓")

## Cell 11: BLEU-4 Evaluation and Visual Inference Check

**Why BLEU-4?**

BLEU-4 (Bilingual Evaluation Understudy, 4-gram) is the standard automatic metric
for image captioning tasks. It measures how much the predicted caption overlaps with
the reference captions using 4-word n-gram precision. It is used by benchmark datasets
such as COCO Captions and is appropriate here since Screen2Words captions follow a
consistent short-phrase style.

**Why 40 samples for evaluation?**

We evaluate on 40 samples from the validation set. This is a lightweight check
suited to the small fine-tuning experiment — sufficient to get a directional BLEU
score without requiring a full dataset evaluation pass.

**Multi-reference evaluation:**

Each image in Screen2Words has 5 human-written captions. We pass all 5 as references
to `corpus_bleu`, which takes the best-matching reference for each n-gram. This is
more fair to the model than single-reference BLEU and is the correct approach for
datasets with multiple valid captions.

**`max_length=32` in `model.generate()`**: Matches the maximum caption length used
during training to keep inference consistent with the training regime.

**Visual inference check:**

We also run inference on a single image from the raw dataset and display it with
`matplotlib`. This provides a qualitative sanity check alongside the quantitative
BLEU score, helping identify whether errors are systematic (e.g. wrong screen type)
or random.

In [ ]:
# Cell-11: Evaluation
!pip install -q nltk
import nltk
nltk.download('punkt_tab')

from nltk.translate.bleu_score import corpus_bleu
import torch
import numpy as np

model.eval()
references = []
hypotheses = []

# Use raw 'small' dataset so we still have original captions
_, val_raw = torch.utils.data.random_split(list(range(len(small))), [400, 100])
val_samples = small.select(list(val_raw.indices)[:40])  # same 40 val samples

print("Sample Predictions:\n")

for i, example in enumerate(val_samples):
    img = example["image"].convert("RGB")
    pixel_values = feature_extractor(
        images=img,
        return_tensors="pt"
    ).pixel_values.to("cuda")

    with torch.no_grad():
        output_ids = model.generate(pixel_values, max_length=32)

    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # All 5 captions as references
    refs = [cap.split() for cap in example["captions"]]
    references.append(refs)
    hypotheses.append(pred.split())

    # Print first 5 samples for visual check
    if i < 5:
        print(f"Image {i+1}")
        print(f"  Predicted : {pred}")
        print(f"  Reference : {example['captions'][0]}")
        print()

bleu = corpus_bleu(references, hypotheses)
print(f"BLEU-4 Score: {bleu:.4f}")

In [ ]:
# Quick Inference on a single image
import matplotlib.pyplot as plt
import torch

# Pick any sample from the raw dataset
sample = dataset["train"][20]
image = sample["image"].convert("RGB")

# Run inference
inputs = feature_extractor(images=image, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    output_ids = model.generate(
        inputs.pixel_values,
        max_length=32
    )

caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Display
plt.imshow(image)
plt.axis("off")
plt.title(f"Predicted: {caption}", wrap=True)
plt.show()

print("Predicted :", caption)
print("Reference :", sample["captions"][0])

In [ ]:
# Cell-12: Upload to HuggingFace
from huggingface_hub import notebook_login

notebook_login()

model.push_to_hub("rico-vit-gpt2-finetuned")
feature_extractor.push_to_hub("rico-vit-gpt2-finetuned")
tokenizer.push_to_hub("rico-vit-gpt2-finetuned")
print("Uploaded to HuggingFace ✓")
print("Model available at: https://huggingface.co/foreseeitwithme/rico-vit-gpt2-finetuned")

## Cell 13: Inference from HuggingFace Hub (Demonstrates Requirement)

This cell fulfills the core project requirement: pulling the fine-tuned model
directly from HuggingFace Hub and running inference — no local files needed.

We load all three components from `foreseeitwithme/rico-vit-gpt2-finetuned`:
- `VisionEncoderDecoderModel.from_pretrained(hf_model_name)` — fine-tuned weights
- `ViTImageProcessor.from_pretrained(hf_model_name)` — image preprocessing config
- `AutoTokenizer.from_pretrained(hf_model_name)` — token decoding config

We call `model.eval()` before inference to disable dropout layers, ensuring
deterministic outputs. `torch.no_grad()` disables gradient tracking to reduce
memory usage during inference — gradients are only needed during training.

We test on samples at indices 0, 5, and 10 to verify the model generalizes across
different UI screen types, not just the first image in the dataset.

In [ ]:
# Cell-13: Inference from Uploaded HuggingFace Model
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
import torch
import matplotlib.pyplot as plt

# Load from HuggingFace
hf_model_name = "foreseeitwithme/rico-vit-gpt2-finetuned"

loaded_model = VisionEncoderDecoderModel.from_pretrained(hf_model_name)
loaded_feature_extractor = ViTImageProcessor.from_pretrained(hf_model_name)
loaded_tokenizer = AutoTokenizer.from_pretrained(hf_model_name)

loaded_model = loaded_model.to("cuda")
loaded_model.eval()

# Test on a few samples
for i in [0, 5, 10]:
    sample = dataset["train"][i]
    image = sample["image"].convert("RGB")

    inputs = loaded_feature_extractor(images=image, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = loaded_model.generate(inputs.pixel_values, max_length=32)

    caption = loaded_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted: {caption}", wrap=True)
    plt.show()

    print(f"Predicted : {caption}")
    print(f"Reference : {sample['captions'][0]}")
    print()

## Results: Sample Predictions and BLEU-4 Evaluation

After fine-tuning on 500 samples from the RICO Screen2Words dataset for 5 epochs,
the model was evaluated on 40 validation samples using corpus BLEU-4.

---

### BLEU-4 Score: 0.1722

BLEU-4 scores for image captioning models typically range from 0.10–0.35 for
single-model systems on domain-specific datasets. A score of 0.1722 is a reasonable
result given:
- Only 500 training samples (full dataset has ~22,000)
- 5 epochs of fine-tuning
- Short captions with high vocabulary variance across screen types

---

### Sample Predictions

| # | Predicted | Reference | Assessment |
|---|-----------|-----------|------------|
| 1 | page displaying drawing image of a girl with long hair | page displaying drawing image of a girl with long hair | Exact match |
| 2 | display of list of options in an app with app settings page | different kinds of rewards and membership details in app |Wrong semantic focus |
| 3 | display of a page showing of workout app with workout app settings page displayed | details of a person in a social app |Wrong app type identified |
| 4 | page asking to enter login details for verification of account details | notification displaying an information |Partial — detected input screen, missed notification context |
| 5 | display of a page showing of payment options for app | page asking to continue with the app |Partial — detected app page, wrong functional context |

---

### Analysis

**What the model gets right:**
- Visually distinctive screens (Image 1 — a drawing) are captioned with high accuracy,
  achieving an exact match with the reference caption.
- The model consistently identifies the correct medium ("display of a page", "app") and
  produces grammatically fluent captions even when the semantic content is wrong.

**Where the model struggles:**
- **Context-dependent screens** (notifications, reward pages, social profiles) are
  frequently misidentified. These screens share visual structure with more common
  screen types (settings, login, options), causing the model to default to
  higher-frequency patterns seen during training.
- **Images 2 and 3** show the model latching onto generic UI patterns ("list of options",
  "settings page") rather than the specific functional content of the screen.
- This is partly a consequence of training on only 500 samples — the model has not
  seen enough diversity to distinguish visually similar screen types reliably.

**Why Image 1 succeeded:**
A drawing of a girl with long hair is visually unambiguous and unlike typical app UI
elements. The model's ViT encoder produces a distinctive feature representation for it,
making caption generation straightforward.

**Why Images 2–5 struggled:**
These screens contain app-specific UI elements (reward badges, profile cards, notification
banners, onboarding CTAs) that look visually similar to generic settings or form screens.
With limited training data, the model cannot reliably learn the fine-grained visual
differences between these screen types.

---

### Key Takeaway

The BLEU-4 score of 0.1722, combined with one exact match and several partially correct
captions, suggests the model has learned the general language style and structure of
Screen2Words captions but needs more training data to reliably capture screen-specific
semantics. Scaling to the full ~22,000 sample dataset would be the most direct path
to improvement.